# Notebook 2 — Bagging, Random Forest & Extra Trees

**Series:** Ensemble Learning · Notebook 2 of 3  
**Theory depth:** [bagging_and_randomization.md](bagging_and_randomization.md)  
**What you build here:** Bootstrap sampling from scratch, BaggingClassifier, RandomForest tuning, feature importance comparison, OOB curve

---

## Section 0 — Setup & Version Check

In [ ]:
# ── All imports up front with version guards ─────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sklearn
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                               ExtraTreesClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

assert sklearn.__version__ >= '1.4', f'Need sklearn >= 1.4, got {sklearn.__version__}'
print(f'sklearn version: {sklearn.__version__}')
np.random.seed(42)

## Section 1 — Quick Theory Recap

Bagging = Bootstrap + Aggregating. Each of m models trains on a different bootstrap sample (~63.2% unique examples).  
Random Forest = Bagging + random feature subset at each split (`max_features='sqrt'`). Reduces inter-tree correlation.  
Extra Trees = RF + random split thresholds. Faster, more diverse, often equally accurate.  
→ Full derivation in [bagging_and_randomization.md](bagging_and_randomization.md)

## Section 2 — Data Preparation

In [ ]:
# ── Load both toy and real datasets ─────────────────────────────────────────
# Toy: high variance, nonlinear, 20 features (10 informative)
X_toy, y_toy = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=4, n_clusters_per_class=2, random_state=42
)
X_toy_tr, X_toy_te, y_toy_tr, y_toy_te = train_test_split(
    X_toy, y_toy, test_size=0.2, stratify=y_toy, random_state=42
)

# Real: Breast Cancer Wisconsin (569 samples, 30 features)
cancer = load_breast_cancer()
X_can, y_can = cancer.data, cancer.target
feat_names = cancer.feature_names
X_tr, X_te, y_tr, y_te = train_test_split(
    X_can, y_can, test_size=0.2, stratify=y_can, random_state=42
)
print(f'Toy   → Train: {X_toy_tr.shape}, Test: {X_toy_te.shape}')
print(f'Cancer → Train: {X_tr.shape}, Test: {X_te.shape}')
print(f'Cancer class balance: {np.bincount(y_tr)}')

## Section 3 — Model Implementation

### 3a. Bootstrap Sampling — From Scratch

In [ ]:
# ── Verify the 63.2% unique-samples property of bootstrap sampling ───────────
rng = np.random.default_rng(42)
n = 1000
n_experiments = 500
pct_unique_list = []

for _ in range(n_experiments):
    # Sample n indices with replacement from {0, ..., n-1}
    sample_indices = rng.integers(0, n, size=n)
    pct_unique = len(set(sample_indices)) / n
    pct_unique_list.append(pct_unique)

print(f'Theoretical: {1 - 1/np.e:.4f} = 1 - 1/e')
print(f'Empirical mean: {np.mean(pct_unique_list):.4f}')
print(f'Empirical std:  {np.std(pct_unique_list):.4f}')
# Confirm: ~63.2% of samples are unique in each bootstrap

In [ ]:
# ── Minimal BaggingClassifier from scratch (to understand the mechanism) ─────
class MinimalBaggingClassifier:
    def __init__(self, n_estimators=10, random_state=42):
        self.n_estimators = n_estimators
        self.rng = np.random.default_rng(random_state)
        self.trees = []

    def fit(self, X, y):
        n = len(X)
        for _ in range(self.n_estimators):
            # Bootstrap: sample with replacement
            idx = self.rng.integers(0, n, size=n)
            tree = DecisionTreeClassifier(max_depth=None, random_state=42)
            tree.fit(X[idx], y[idx])  # each tree sees a different slice of data
            self.trees.append(tree)
        return self

    def predict(self, X):
        # Majority vote across all trees
        votes = np.column_stack([t.predict(X) for t in self.trees])
        return np.apply_along_axis(lambda row: np.bincount(row).argmax(), 1, votes)

# Compare: single tree vs minimal bagging vs sklearn BaggingClassifier
single_tree = DecisionTreeClassifier(max_depth=None, random_state=42)
single_tree.fit(X_toy_tr, y_toy_tr)

my_bag = MinimalBaggingClassifier(n_estimators=100, random_state=42)
my_bag.fit(X_toy_tr, y_toy_tr)

sk_bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None, random_state=42),
    n_estimators=100, bootstrap=True, oob_score=True, random_state=42, n_jobs=-1
)
sk_bag.fit(X_toy_tr, y_toy_tr)

print(f'Single Tree accuracy:       {accuracy_score(y_toy_te, single_tree.predict(X_toy_te)):.4f}')
print(f'Minimal Bagging accuracy:   {accuracy_score(y_toy_te, my_bag.predict(X_toy_te)):.4f}')
print(f'sklearn Bagging accuracy:   {accuracy_score(y_toy_te, sk_bag.predict(X_toy_te)):.4f}')
print(f'sklearn OOB score:          {sk_bag.oob_score_:.4f}  ← free validation!')

In [ ]:
# ── Random Forest and Extra Trees on both datasets ───────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',  # NOT 'auto' — deprecated in sklearn 1.1+
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
et = ExtraTreesClassifier(
    n_estimators=200,
    max_features='sqrt',
    oob_score=True,
    bootstrap=True,  # must set True for OOB to be available in ExtraTrees
    random_state=42,
    n_jobs=-1
)

rf.fit(X_tr, y_tr)
et.fit(X_tr, y_tr)

print(f'Random Forest  → Test Acc: {accuracy_score(y_te, rf.predict(X_te)):.4f} | OOB: {rf.oob_score_:.4f}')
print(f'Extra Trees    → Test Acc: {accuracy_score(y_te, et.predict(X_te)):.4f} | OOB: {et.oob_score_:.4f}')

## Section 4 — Hyperparameter Deep-Dive

In [ ]:
# ── Effect of n_estimators on OOB score (to show diminishing returns) ────────
estimator_counts = [10, 25, 50, 100, 150, 200, 300, 400, 500]
oob_scores_rf = []

for n in estimator_counts:
    model = RandomForestClassifier(
        n_estimators=n, max_features='sqrt',
        oob_score=True, random_state=42, n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    oob_scores_rf.append(model.oob_score_)
    print(f'n_estimators={n:>4} → OOB score: {model.oob_score_:.4f}')

In [ ]:
# ── Effect of max_features on RF accuracy ────────────────────────────────────
# This is the diversity dial: lower = more diversity, potentially better ensemble
p = X_tr.shape[1]  # number of features
max_features_options = {
    'All features (p)': p,
    'sqrt(p)': 'sqrt',
    'log2(p)': 'log2',
    '1 feature': 1,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for label, mf in max_features_options.items():
    model = RandomForestClassifier(n_estimators=200, max_features=mf, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X_can, y_can, cv=cv, scoring='accuracy')
    print(f'{label:>25}: mean={scores.mean():.4f} ± {scores.std():.4f}')

## Section 5 — Visualizations

In [ ]:
# ── Plot OOB error curve and feature importance comparison ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. OOB score vs n_estimators
ax = axes[0]
ax.plot(estimator_counts, oob_scores_rf, 'o-', color='#2196F3')
ax.axhline(oob_scores_rf[-1], color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Number of Trees')
ax.set_ylabel('OOB Score')
ax.set_title('Diminishing Returns: OOB Score vs n_estimators')
ax.grid(alpha=0.3)

# 2. MDI Feature Importance (top 15)
ax2 = axes[1]
rf_full = RandomForestClassifier(n_estimators=300, max_features='sqrt', random_state=42, n_jobs=-1)
rf_full.fit(X_tr, y_tr)
mdi_imp = pd.Series(rf_full.feature_importances_, index=feat_names)
mdi_imp.nlargest(15).sort_values().plot(kind='barh', ax=ax2, color='#4CAF50')
ax2.set_title('MDI Feature Importance (top 15)')
ax2.set_xlabel('Importance (can be biased for high-cardinality features)')

# 3. Permutation Feature Importance (unbiased)
ax3 = axes[2]
perm_result = permutation_importance(
    rf_full, X_te, y_te, n_repeats=15, random_state=42, n_jobs=-1
)
perm_imp = pd.Series(perm_result.importances_mean, index=feat_names)
perm_imp.nlargest(15).sort_values().plot(kind='barh', ax=ax3, color='#FF9800')
ax3.set_title('Permutation Importance (top 15, unbiased)')
ax3.set_xlabel('Mean accuracy decrease when feature shuffled')

plt.tight_layout()
plt.savefig('rf_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices: single tree vs Random Forest ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

single = DecisionTreeClassifier(random_state=42)
single.fit(X_tr, y_tr)

for ax, model, name in zip(axes, [single, rf_full], ['Single Decision Tree', 'Random Forest (300 trees)']):
    ConfusionMatrixDisplay.from_estimator(
        model, X_te, y_te,
        display_labels=cancer.target_names,
        cmap='Blues', ax=ax, colorbar=False
    )
    ax.set_title(f'{name}\nAcc: {accuracy_score(y_te, model.predict(X_te)):.4f}')

plt.tight_layout()
plt.savefig('confusion_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 6 — Comparison Table

In [ ]:
# ── Full comparison: bagging variants on breast cancer ───────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Decision Tree (baseline)':   DecisionTreeClassifier(random_state=42),
    'Bagging (100 trees)':        BaggingClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1),
    'Random Forest (200 trees)':  RandomForestClassifier(n_estimators=200, max_features='sqrt', oob_score=True, random_state=42, n_jobs=-1),
    'Extra Trees (200 trees)':    ExtraTreesClassifier(n_estimators=200, max_features='sqrt', bootstrap=True, oob_score=True, random_state=42, n_jobs=-1),
}

rows = []
for name, model in models.items():
    model.fit(X_tr, y_tr)
    test_acc = accuracy_score(y_te, model.predict(X_te))
    cv_scores = cross_val_score(model, X_can, y_can, cv=cv)
    oob = getattr(model, 'oob_score_', None)
    rows.append({
        'Model': name,
        'Test Acc': round(test_acc, 4),
        'CV Mean': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4),
        'OOB Score': round(oob, 4) if oob else 'N/A'
    })

df = pd.DataFrame(rows)
df.style.highlight_max(subset=['Test Acc', 'CV Mean'], color='lightgreen') \
        .highlight_min(subset=['CV Std'], color='lightyellow') \
        .set_caption('Breast Cancer Wisconsin — Bagging Variants Comparison')

## Section 7 — Key Takeaways

- **Bootstrap uniqueness is ~63.2%** — empirically verified here. The rest become free OOB validation samples.
- **Feature randomness (`max_features`) is the most important RF hyperparameter** — it controls inter-tree correlation, which is the bottleneck on variance reduction.
- **MDI vs Permutation importance:** MDI is fast but biased toward high-cardinality features. Always use permutation importance before dropping features based on RF importance.
- **Extra Trees trains faster** (no threshold search) and often matches RF accuracy — always benchmark both before committing to RF.
- **Adding more trees never hurts in bagging** (unlike boosting) — use `n_jobs=-1` to exploit parallelism and go up to 500+ trees for important models.